# MotoGP Riders Info - Dataset Exploration

This notebook explores the all-time rider statistics dataset covering victories, podiums, pole positions, fastest laps, and world championships from 1974 to 2022.

## 0. Notebook Setup

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Loading

Let's load the riders_info.csv file containing all-time rider statistics.

In [ ]:
# Load data from CSV
data_path = Path("../data/raw/riders_info.csv")
df = pd.read_csv(data_path)

print(f"Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

## 2. Data Structure Exploration

Let's examine the data structure and understand what each column represents.

In [ ]:
print(f"Dimensions: {df.shape}")
print("-" * 50)
print(f"Columns: {list(df.columns)}")
print("-" * 50)
print(f"\nData types:")
print(df.dtypes)
print("-" * 50)
print(f"\nNull values:")
print(df.isnull().sum())

In [ ]:
# Get basic statistical information
print("=== BASIC STATISTICS ===")
df.describe()

In [ ]:
# Check for missing values in detail
print("=== MISSING VALUES ANALYSIS ===")
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100
missing_table = pd.DataFrame({
    'Missing Count': missing_data,
    'Percentage': missing_percentage
})
print(missing_table[missing_table['Missing Count'] > 0])

## 3. Top Performers Analysis

Let's analyze the top performers across different metrics.

In [ ]:
# Top 10 riders by victories
print("=== TOP 10 RIDERS BY VICTORIES ===")
top_victories = df.nlargest(10, 'Victories')[['Riders All Time in All Classes', 'Victories', 'World Championships']]
print(top_victories.to_string(index=False))

In [ ]:
# Top 10 riders by world championships
print("=== TOP 10 RIDERS BY WORLD CHAMPIONSHIPS ===")
top_championships = df.nlargest(10, 'World Championships')[['Riders All Time in All Classes', 'World Championships', 'Victories']]
print(top_championships.to_string(index=False))

In [ ]:
# Top 10 riders by pole positions (handling missing values)
print("=== TOP 10 RIDERS BY POLE POSITIONS (from 1974) ===")
df_poles = df.dropna(subset=['Pole positions from \'74 to 2022'])
top_poles = df_poles.nlargest(10, 'Pole positions from \'74 to 2022')[['Riders All Time in All Classes', 'Pole positions from \'74 to 2022', 'Victories']]
print(top_poles.to_string(index=False))

In [ ]:
# Top 10 riders by fastest laps
print("=== TOP 10 RIDERS BY FASTEST LAPS (to 2022) ===")
df_fastest = df.dropna(subset=['Race fastest lap to 2022'])
top_fastest = df_fastest.nlargest(10, 'Race fastest lap to 2022')[['Riders All Time in All Classes', 'Race fastest lap to 2022', 'Victories']]
print(top_fastest.to_string(index=False))

## 4. Podium Analysis

Let's analyze podium statistics (1st, 2nd, 3rd places).

In [ ]:
# Calculate total podiums
df['Total Podiums'] = df['Victories'] + df['2nd places'].fillna(0) + df['3rd places'].fillna(0)

print("=== TOP 10 RIDERS BY TOTAL PODIUMS ===")
top_podiums = df.nlargest(10, 'Total Podiums')[['Riders All Time in All Classes', 'Total Podiums', 'Victories', '2nd places', '3rd places']]
print(top_podiums.to_string(index=False))

In [ ]:
# Calculate podium efficiency (victories / total podiums)
df['Podium Win Rate'] = df['Victories'] / df['Total Podiums']

print("=== PODIUM WIN RATE ANALYSIS (min 10 podiums) ===")
df_min_podiums = df[df['Total Podiums'] >= 10]
top_win_rate = df_min_podiums.nlargest(10, 'Podium Win Rate')[['Riders All Time in All Classes', 'Podium Win Rate', 'Victories', 'Total Podiums']]
print(top_win_rate.to_string(index=False))

## 5. Visualizations

Let's create visualizations to better understand the data patterns.

In [ ]:
# Bar chart: Top 15 riders by victories
plt.figure(figsize=(12, 8))
top_15_victories = df.nlargest(15, 'Victories')
plt.barh(range(len(top_15_victories)), top_15_victories['Victories'], color='skyblue')
plt.yticks(range(len(top_15_victories)), top_15_victories['Riders All Time in All Classes'])
plt.xlabel('Number of Victories')
plt.title('Top 15 Riders by Victories (All Classes, All Time)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Victories vs World Championships
plt.figure(figsize=(12, 8))
plt.scatter(df['Victories'], df['World Championships'], alpha=0.7, s=60)

# Add labels for top performers
top_performers = df.nlargest(8, 'Victories')
for idx, row in top_performers.iterrows():
    plt.annotate(row['Riders All Time in All Classes'].split()[0], 
                (row['Victories'], row['World Championships']),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel('Victories')
plt.ylabel('World Championships')
plt.title('Victories vs World Championships')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of victories
plt.figure(figsize=(12, 6))
plt.hist(df['Victories'], bins=30, alpha=0.7, color='lightcoral', edgecolor='black')
plt.xlabel('Number of Victories')
plt.ylabel('Number of Riders')
plt.title('Distribution of Victories Among All Riders')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numerical columns
plt.figure(figsize=(10, 8))
numeric_cols = ['Victories', '2nd places', '3rd places', 'Pole positions from \'74 to 2022', 
                'Race fastest lap to 2022', 'World Championships']
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f')
plt.title('Correlation Matrix of Rider Performance Metrics')
plt.tight_layout()
plt.show()

## 6. Performance Categories Analysis

Let's categorize riders based on their performance levels.

In [ ]:
# Create performance categories based on victories
def categorize_performance(victories):
    if victories >= 50:
        return 'Legend (50+ victories)'
    elif victories >= 20:
        return 'Elite (20-49 victories)'
    elif victories >= 5:
        return 'Competitive (5-19 victories)'
    elif victories >= 1:
        return 'Winner (1-4 victories)'
    else:
        return 'No victories'

df['Performance Category'] = df['Victories'].apply(categorize_performance)

print("=== PERFORMANCE CATEGORIES ===")
category_counts = df['Performance Category'].value_counts()
print(category_counts)

# Pie chart of performance categories
plt.figure(figsize=(10, 8))
plt.pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%', 
        startangle=90, colors=plt.cm.Set3.colors)
plt.title('Distribution of Riders by Performance Category')
plt.axis('equal')
plt.tight_layout()
plt.show()

## 7. Efficiency Metrics

Let's analyze various efficiency metrics for top riders.

In [ ]:
# Championships per victory ratio for riders with multiple championships
df_champions = df[df['World Championships'] > 0].copy()
df_champions['Championships per Victory'] = df_champions['World Championships'] / df_champions['Victories']

print("=== CHAMPIONSHIP EFFICIENCY (Championships/Victories) ===")
efficiency_analysis = df_champions.nlargest(10, 'Championships per Victory')[[
    'Riders All Time in All Classes', 'Championships per Victory', 
    'World Championships', 'Victories'
]]
print(efficiency_analysis.to_string(index=False))

In [ ]:
# Pole position to victory conversion rate
df_poles_clean = df.dropna(subset=['Pole positions from \'74 to 2022']).copy()
df_poles_clean = df_poles_clean[df_poles_clean['Pole positions from \'74 to 2022'] > 0]
df_poles_clean['Pole to Win Rate'] = df_poles_clean['Victories'] / df_poles_clean['Pole positions from \'74 to 2022']

print("=== POLE TO WIN CONVERSION RATE (min 10 poles) ===")
df_min_poles = df_poles_clean[df_poles_clean['Pole positions from \'74 to 2022'] >= 10]
pole_conversion = df_min_poles.nlargest(10, 'Pole to Win Rate')[[
    'Riders All Time in All Classes', 'Pole to Win Rate', 
    'Victories', 'Pole positions from \'74 to 2022'
]]
print(pole_conversion.to_string(index=False))

## 8. Key Insights Summary

Based on the exploration of the riders_info dataset:

### Dataset Characteristics
- Contains career statistics for motorcycle racing legends
- Covers victories, podiums, pole positions (from 1974), fastest laps, and championships
- Some missing data in pole positions and fastest laps for older riders

### Top Performers
- **Most Victories**: Giacomo Agostini leads with 122 victories
- **Most Championships**: Agostini also leads in world championships
- **Modern Era**: Valentino Rossi and Marc Marquez dominate recent statistics

### Performance Patterns
- Strong positive correlation between victories and championships
- Elite riders show consistent performance across all metrics
- Championship efficiency varies significantly among top riders

### Data Quality Notes
- Pole position data only available from 1974 onwards
- Some riders have missing data in certain categories
- Need to cross-reference with other datasets for complete analysis

This exploration provides the foundation for answering specific business questions about rider performance, fastest laps, and championship statistics.